# 07 – Fine-tune Transformer + baseline cổ điển (Elden Ring Steam Reviews)

Notebook bổ sung cho đồ án Khai phá dữ liệu – Vũ Đức Toàn (23010062).

**Chạy trên Google Colab:** Runtime → Change runtime type → **T4 GPU**, rồi Run all (khoảng 10–15 phút cho 3 seed với DistilBERT).

Nội dung:
1. Tải dữ liệu đã chia sẵn từ repo (cùng Train/Validation/Test với các notebook 01–06).
2. Baseline: đa số, TF-IDF + LR (tái hiện kết quả cũ), Linear SVM, Complement NB.
3. Fine-tune DistilBERT (hoặc RoBERTa) với loss có trọng số lớp, 3 seed, chọn epoch theo Macro-F1 trên Validation.
4. Đánh giá Test, khoảng tin cậy bootstrap, kiểm định McNemar, phân tích lỗi, và in bảng sẵn để dán vào báo cáo.

**Quy ước chống rò rỉ:** mô hình/epoch chỉ chọn theo Validation; Test chỉ dùng để dự đoán sau khi chọn.

**Kết quả CSV:** lưu vào `repo/results/`; sau khi chạy xong, tải về thư mục `results/` của repo.

In [ ]:
!pip -q install -U "transformers>=5,<6" "accelerate>=1.1.0" scikit-learn scipy

In [ ]:
import os, subprocess, random, math, numpy as np, pandas as pd, torch
import transformers, sklearn, scipy
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, confusion_matrix

print("transformers", transformers.__version__, "| torch", torch.__version__, "| sklearn", sklearn.__version__, "| scipy", scipy.__version__)

# ====== CẤU HÌNH ======
REPO = "https://github.com/23010062-star/Elden-Ring-Sentiment-Analysis.git"
MODEL_NAME = "distilbert-base-uncased"   # có thể đổi sang "roberta-base" rồi chạy lại
SEEDS = [42, 43, 44]
MAX_LEN, EPOCHS, LR, BATCH = 128, 3, 2e-5, 32
TEXT_COL, LABEL_COL = "review", "label"  # sửa nếu tên cột trong CSV khác (xem kết quả in bên dưới)

In [ ]:
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "repo"], check=True)
D = "repo/data/processed"
# Nếu CSV là con trỏ Git LFS (file rất nhỏ) thì kéo dữ liệu thật
if os.path.getsize(f"{D}/train.csv") < 2000:
    subprocess.run(["git", "lfs", "pull"], cwd="repo")
RESULTS_DIR = "repo/results"
os.makedirs(RESULTS_DIR, exist_ok=True)
tr, va, te = [pd.read_csv(f"{D}/{n}.csv") for n in ["train", "validation", "test"]]
print("Cột:", tr.columns.tolist())
assert TEXT_COL in tr.columns and LABEL_COL in tr.columns, "Hãy sửa TEXT_COL / LABEL_COL theo danh sách cột ở trên"
for d in (tr, va, te):
    d[TEXT_COL] = d[TEXT_COL].fillna("").astype(str)
    d[LABEL_COL] = d[LABEL_COL].astype(int)
print(len(tr), len(va), len(te))
assert (len(tr), len(va), len(te)) == (5562, 1192, 1192), "Kích thước tập khác báo cáo – kiểm tra notebook 03 đã chạy chưa"
print(te[LABEL_COL].value_counts().to_dict())
ytr, yva, yte = tr[LABEL_COL].values, va[LABEL_COL].values, te[LABEL_COL].values

In [ ]:
def full_metrics(y, p):
    y, p = np.asarray(y), np.asarray(p)
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, labels=[1], zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return dict(acc=accuracy_score(y, p), prec=pr[0], rec=rc[0], f1=f1[0],
                macro_f1=f1_score(y, p, average="macro", labels=[0, 1]), f1_0=f1_score(y, p, pos_label=0, zero_division=0),
                TP=int(tp), FN=int(fn), FP=int(fp), TN=int(tn),
                prec0=tn / (tn + fn) if tn + fn else 0.0, rec0=tn / (tn + fp) if tn + fp else 0.0)

fmt = lambda v, n=4: f"{v:.{n}f}".replace(".", ",")

## 1. Baseline (đa số, TF-IDF + LR, Linear SVM, Complement NB)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.dummy import DummyClassifier

vec = TfidfVectorizer()
Xtr = vec.fit_transform(tr[TEXT_COL]); Xva = vec.transform(va[TEXT_COL]); Xte = vec.transform(te[TEXT_COL])
base = {
    "Baseline đa số": DummyClassifier(strategy="most_frequent"),
    "TF-IDF + LR (balanced)": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "TF-IDF + Linear SVM (balanced)": LinearSVC(class_weight="balanced"),
    "TF-IDF + Complement NB": ComplementNB(),
}
R, PT = {}, {}   # R: kết quả, PT: dự đoán Test
for name, m in base.items():
    m.fit(Xtr, ytr)
    R[name] = dict(val=full_metrics(yva, m.predict(Xva)), test=full_metrics(yte, m.predict(Xte)))
    PT[name] = m.predict(Xte)
    print(f"{name:34s} Val Macro-F1={R[name]['val']['macro_f1']:.4f} | Test Macro-F1={R[name]['test']['macro_f1']:.4f}, F1 lớp 0={R[name]['test']['f1_0']:.4f}")
chk = R["TF-IDF + LR (balanced)"]["test"]["macro_f1"]
print("\nKiểm tra tái hiện: Macro-F1 Test của TF-IDF + LR phải ≈ 0,7124 ->", round(chk, 4), "OK" if abs(chk - 0.7124) < 0.002 else "LỆCH – kiểm tra dữ liệu/phiên bản")

## 2. Fine-tune Transformer
Loss là cross-entropy có trọng số lớp `n / (K · n_c)` — giống `class_weight="balanced"` của Logistic Regression để so sánh công bằng. Epoch tốt nhất được chọn theo **Macro-F1 trên Validation**.

In [ ]:
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, set_seed)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
class DS(Dataset):
    def __init__(self, texts, labels):
        self.enc = tok(list(texts), truncation=True, max_length=MAX_LEN); self.y = list(labels)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        d = {k: v[i] for k, v in self.enc.items()}; d["labels"] = int(self.y[i]); return d
ds_tr, ds_va, ds_te = DS(tr[TEXT_COL], ytr), DS(va[TEXT_COL], yva), DS(te[TEXT_COL], yte)

cnt = np.bincount(ytr)
W = torch.tensor(len(ytr) / (2 * cnt), dtype=torch.float)
print("Trọng số lớp (0, 1):", W.tolist())      # kỳ vọng ≈ (7,06; 0,54)

class WTrainer(Trainer):
    loss_is_scaled_for_ga = False  # loss của ta là trung bình theo batch
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, labels, weight=W.to(device=out.logits.device, dtype=out.logits.dtype))
        return (loss, out) if return_outputs else loss

def compute_metrics(ep):
    lg, lb = ep
    return {"macro_f1": f1_score(lb, lg.argmax(-1), average="macro", labels=[0, 1])}

In [ ]:
warmup_steps = max(1, int(0.1 * math.ceil(len(ds_tr) / BATCH) * EPOCHS))
print("Warm-up steps:", warmup_steps)
runs = []
for seed in SEEDS:
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    args = TrainingArguments(
        output_dir=f"out/{seed}", num_train_epochs=EPOCHS, learning_rate=LR,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=64,
        weight_decay=0.01, warmup_steps=warmup_steps, eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
        save_total_limit=1, seed=seed, fp16=torch.cuda.is_available(), report_to="none")
    t = WTrainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
                 data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
    t.train()
    pv = t.predict(ds_va).predictions.argmax(-1)      # epoch tốt nhất đã chọn theo Validation
    pt = t.predict(ds_te).predictions.argmax(-1)      # Test chỉ dự đoán sau khi đã chọn
    runs.append(dict(seed=seed, val=full_metrics(yva, pv), test=full_metrics(yte, pt), pt=pt))
    print(f"seed {seed}: Val Macro-F1={runs[-1]['val']['macro_f1']:.4f} | Test Macro-F1={runs[-1]['test']['macro_f1']:.4f}, F1 lớp 0={runs[-1]['test']['f1_0']:.4f}")
    del model, t; torch.cuda.empty_cache()

## 3. Tổng hợp, khoảng tin cậy và kiểm định

In [ ]:
keys = ["acc", "prec", "rec", "f1", "macro_f1", "f1_0", "prec0", "rec0"]
tm = pd.DataFrame([r["test"] for r in runs])[keys + ["TP", "FN", "FP", "TN"]]
vm = pd.DataFrame([r["val"] for r in runs])[["macro_f1", "f1_0"]]
print("Test theo từng seed:"); print(tm.round(4).to_string())
print("\nMean ± std (Test):"); print(pd.DataFrame({"mean": tm[keys].mean(), "std": tm[keys].std(ddof=1)}).round(4).T.to_string())
print("\nValidation Macro-F1 mean ± std:", round(vm.macro_f1.mean(), 4), "±", round(vm.macro_f1.std(ddof=1), 4))

# --- Bootstrap CI cho Macro-F1 trên Test (seed đầu tiên) và hiệu số so với TF-IDF + LR ---
from scipy.stats import binomtest
pt0 = runs[0]["pt"]; ptf = PT["TF-IDF + LR (balanced)"]
rng = np.random.default_rng(0); B = 2000; n = len(yte)
mf = lambda y, p: f1_score(y, p, average="macro", labels=[0, 1])
bs_b, bs_d = [], []
for _ in range(B):
    i = rng.integers(0, n, n)
    a, b = mf(yte[i], pt0[i]), mf(yte[i], ptf[i]); bs_b.append(a); bs_d.append(a - b)
lo, hi = np.percentile(bs_b, [2.5, 97.5]); dlo, dhi = np.percentile(bs_d, [2.5, 97.5])
print(f"\nMacro-F1 Test (seed {SEEDS[0]}) 95% CI: [{lo:.4f}, {hi:.4f}]")
print(f"Hiệu số Macro-F1 (Transformer − TF-IDF+LR) 95% CI: [{dlo:+.4f}, {dhi:+.4f}]  (CI chứa 0 ⇒ chưa đủ bằng chứng khác biệt)")

# --- McNemar chính xác ---
ct, cf = (pt0 == yte), (ptf == yte)
b01, c10 = int((cf & ~ct).sum()), int((~cf & ct).sum())   # TF-IDF đúng/Transformer sai ; ngược lại
p = binomtest(min(b01, c10), b01 + c10, 0.5).pvalue if b01 + c10 else 1.0
print(f"McNemar: TF-IDF đúng & Transformer sai = {b01}, ngược lại = {c10}, p = {p:.4f}")

In [ ]:
# --- Phân tích lỗi (seed đầu tiên): lưu và in ví dụ review bị phân loại sai ---
os.makedirs(RESULTS_DIR, exist_ok=True)
e = te.assign(pred=pt0, pred_tfidf=ptf, seed=SEEDS[0])
miss_neg = e[(e[LABEL_COL] == 0) & (e.pred == 1)]   # review KHÔNG đề xuất bị đoán là đề xuất
false_neg = e[(e[LABEL_COL] == 1) & (e.pred == 0)]  # review ĐỀ XUẤT bị đoán là không đề xuất
print("Lớp 0 → 1 (bỏ sót lớp 0):", len(miss_neg), "| Lớp 1 → 0 (báo nhầm lớp 0):", len(false_neg))
show = lambda d, k=12: [print("-", s[:220].replace("\n", " ")) for s in d[TEXT_COL].head(k)]
print("\n== Review KHÔNG đề xuất nhưng mô hình đoán 'đề xuất' =="); show(miss_neg)
print("\n== Review ĐỀ XUẤT nhưng mô hình đoán 'không đề xuất' =="); show(false_neg)
e[e[LABEL_COL] != e.pred].to_csv(f"{RESULTS_DIR}/errors_{MODEL_NAME.split('/')[-1]}.csv", index=False)

In [ ]:
# --- In dòng/bảng sẵn để dán vào báo cáo (định dạng số kiểu Việt Nam) ---
print("BẢNG 4 (Test): Phương pháp | Acc | Prec | Recall | F1 | Macro-F1 | F1 lớp 0")
for name in base:
    m = R[name]["test"]
    print(" | ".join([name] + [fmt(m[k]) for k in ["acc", "prec", "rec", "f1", "macro_f1", "f1_0"]]))
mu, sd = tm[keys].mean(), tm[keys].std(ddof=1)
label = MODEL_NAME.split("/")[-1] + " fine-tune (TB ± độ lệch, 3 seed)"
print(" | ".join([label] + [f"{fmt(mu[k])} ± {fmt(sd[k])}" for k in ["acc", "prec", "rec", "f1", "macro_f1", "f1_0"]]))
print("\nBẢNG 5 (seed", SEEDS[0], "): TP | FN | FP | TN | Precision lớp 0 | Recall lớp 0")
m = runs[0]["test"]; print(" | ".join([str(m[k]) for k in ["TP", "FN", "FP", "TN"]] + [fmt(m["prec0"]), fmt(m["rec0"])]))
out = pd.DataFrame({n_: {k: R[n_]["test"][k] for k in keys} for n_ in base}).T
out.loc[label + " (mean)"] = mu.reindex(out.columns)
out.to_csv(f"{RESULTS_DIR}/fine_tune_results_{MODEL_NAME.split('/')[-1]}.csv"); print(f"\nĐã lưu {RESULTS_DIR}/*.csv (tải về từ thanh bên trái Colab)")